# 01 — Raw Data Profiling: NHS RTT Full Extracts

**Layer:** Bronze (raw)  
**Objective:** Profile the raw monthly NHS Referral-to-Treatment (RTT) full extracts prior to ingestion — establishing schema, grain, a data-quality baseline, and cross-period schema consistency.  
**Inputs:** `data/raw/*-RTT-*-full-extract*.csv` — monthly wide-format aggregate extracts (provider × commissioner × pathway type × treatment function, with weekly wait-band counts).  
**Outputs:** None written. Findings inform the Silver-layer validation and cleansing specification.  
**Method:** Inventory extracts → profile a representative period → classify columns → assess data quality → verify schema stability across all periods → build a lightweight combined view.

Source extracts are treated as immutable; this notebook is read-only with respect to `data/raw/`.

**Execution:** select the `Python (HCIP)` kernel, then run all cells in order.

## 1. Configuration

In [2]:
import re
from pathlib import Path

import pandas as pd


def resolve_raw_dir() -> Path:
    """Locate the data/raw directory from any working directory within the project."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "data" / "raw"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not locate a 'data/raw' directory above the working directory.")


RAW_DIR = resolve_raw_dir()
EXTRACT_GLOB = "*-RTT-*-full-extract*.csv"

# Grain at which a single extract is expected to be unique.
GRAIN = [
    "Period",
    "Provider Org Code",
    "Commissioner Org Code",
    "RTT Part Type",
    "Treatment Function Code",
]

# Non-wait-band measure columns appended after the weekly bands.
SUMMARY_MEASURES = ["Total", "Patients with unknown clock start date", "Total All"]

# Lightweight column subset for the combined cross-period view (section 8).
COMBINE_COLS = [
    "Period",
    "Provider Org Code",
    "Provider Org Name",
    "Treatment Function Code",
    "Treatment Function Name",
    "RTT Part Description",
    "Total",
    "Total All",
]

## 2. Extract inventory

Enumerate the monthly extracts available in the raw layer and parse the reporting period from each filename's `YYYYMMDD` prefix (the period end date).

In [3]:
extracts = sorted(RAW_DIR.glob(EXTRACT_GLOB))
if not extracts:
    raise FileNotFoundError(f"No extracts matching '{EXTRACT_GLOB}' in {RAW_DIR}.")

inventory = pd.DataFrame(
    {
        "file": [p.name for p in extracts],
        "period_end": pd.to_datetime(
            [re.match(r"(\d{8})", p.name).group(1) for p in extracts], format="%Y%m%d"
        ),
        "size_mb": [round(p.stat().st_size / 1_000_000, 1) for p in extracts],
    }
).sort_values("period_end", ignore_index=True)

print(f"{len(extracts)} monthly extracts spanning "
      f"{inventory['period_end'].min():%Y-%m} to {inventory['period_end'].max():%Y-%m}")
inventory

24 monthly extracts spanning 2024-04 to 2026-03


,file,period_end,size_mb
0,20240430-RTT-April-2024-full-extract-revised.csv,2024-04-30,84.3
1,20240531-RTT-May-2024-full-extract-revised.csv,2024-05-31,83.8
2,20240630-RTT-June-2024-full-extract-revised.csv,2024-06-30,83.7
3,20240731-RTT-July-2024-full-extract-revised.csv,2024-07-31,85.4
4,20240831-RTT-August-2024-full-extract-revised.csv,2024-08-31,84.7
5,20240930-RTT-September-2024-full-extract-revis...,2024-09-30,86.1
6,20241031-RTT-October-2024-full-extract-revised...,2024-10-31,87.9
7,20241130-RTT-November-2024-full-extract-revise...,2024-11-30,86.8
8,20241231-RTT-December-2024-full-extract-revise...,2024-12-31,84.6
9,20250131-RTT-January-2025-full-extract-revised...,2025-01-31,87.1


## 3. Representative period

Load the most recent extract in full to profile schema and content. Remaining periods share this schema (verified in section 7).

In [4]:
sample_file = extracts[-1]

try:
    df = pd.read_csv(sample_file, low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(sample_file, low_memory=False, encoding="latin-1")

print(f"Profiled extract: {sample_file.name}")
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")

Profiled extract: 20260331-RTT-March-2026-full-extract.csv
Rows: 183,400   Columns: 121


## 4. Column classification

Partition columns into identifiers (dimensional attributes), weekly wait-band measures (counts of pathways in each `>n–n+1 weeks` bucket), and summary measures.

In [5]:
wait_band_cols = [c for c in df.columns if c.endswith("SUM 1")]
summary_cols = [c for c in SUMMARY_MEASURES if c in df.columns]
identifier_cols = [c for c in df.columns if c not in wait_band_cols + summary_cols]

print(f"Identifier columns:   {len(identifier_cols)}")
print(f"Wait-band measures:   {len(wait_band_cols)}  ({wait_band_cols[0]} ... {wait_band_cols[-1]})")
print(f"Summary measures:     {summary_cols}")
df[identifier_cols].dtypes

Identifier columns:   13
Wait-band measures:   105  (Gt 00 To 01 Weeks SUM 1 ... Gt 104 Weeks SUM 1)
Summary measures:     ['Total', 'Patients with unknown clock start date', 'Total All']


Period                          str
Provider Parent Org Code        str
Provider Parent Name            str
Provider Org Code               str
Provider Org Name               str
Commissioner Parent Org Code    str
Commissioner Parent Name        str
Commissioner Org Code           str
Commissioner Org Name           str
RTT Part Type                   str
RTT Part Description            str
Treatment Function Code         str
Treatment Function Name         str
dtype: object

## 5. Sample records

Identifier attributes and summary measures for the first 20 rows.

In [6]:
df[identifier_cols + summary_cols].head(20)

,Period,Provider Parent Org Code,Provider Parent Name,Provider Org Code,Provider Org Name,Commissioner Parent Org Code,Commissioner Parent Name,Commissioner Org Code,Commissioner Org Name,RTT Part Type,RTT Part Description,Treatment Function Code,Treatment Function Name,Total,Patients with unknown clock start date,Total All
0,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,Incomplete Pathways,C_100,General Surgery Service,NaN,NaN,1
1,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,Incomplete Pathways,C_101,Urology Service,NaN,NaN,1
2,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,Incomplete Pathways,C_999,Total,NaN,NaN,2
3,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_3,New RTT Periods - All Patients,C_100,General Surgery Service,NaN,NaN,1
4,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_3,New RTT Periods - All Patients,C_999,Total,NaN,NaN,1
5,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,32T,NORTH WEST - H&J COMMISSIONING HUB,Part_2,Incomplete Pathways,C_100,General Surgery Service,NaN,NaN,1
6,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,32T,NORTH WEST - H&J COMMISSIONING HUB,Part_2,Incomplete Pathways,C_999,Total,NaN,NaN,1
7,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,32T,NORTH WEST - H&J COMMISSIONING HUB,Part_3,New RTT Periods - All Patients,C_100,General Surgery Service,NaN,NaN,1
8,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,32T,NORTH WEST - H&J COMMISSIONING HUB,Part_3,New RTT Periods - All Patients,C_999,Total,NaN,NaN,1
9,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,Y62,NaN,Part_2,Incomplete Pathways,C_101,Urology Service,NaN,NaN,1


## 6. Dimensional cardinality and data quality

Distinct counts for key dimensions, the distribution of pathway types, grain uniqueness, and missing-value counts.

In [7]:
for col in ["Provider Org Name", "Commissioner Org Name", "Treatment Function Name", "RTT Part Description"]:
    if col in df.columns:
        print(f"{col:<28} {df[col].nunique():>6} distinct")

print("\nPathway type distribution:")
print(df["RTT Part Description"].value_counts())

Provider Org Name               532 distinct
Commissioner Org Name           121 distinct
Treatment Function Name          24 distinct
RTT Part Description              5 distinct

Pathway type distribution:
RTT Part Description
Incomplete Pathways                             63547
New RTT Periods - All Patients                  37271
Completed Pathways For Non-Admitted Patients    33387
Incomplete Pathways with DTA                    30311
Completed Pathways For Admitted Patients        18884
Name: count, dtype: int64


In [8]:
duplicate_rows = df.duplicated(subset=GRAIN).sum()
print(f"Rows:                         {len(df):,}")
print(f"Duplicates at declared grain: {duplicate_rows:,}")

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nColumns with missing values:  {len(missing)}")
missing.head(20)

Rows:                         183,400
Duplicates at declared grain: 0

Columns with missing values:  110


Patients with unknown clock start date    149005
Total                                     131129
Gt 103 To 104 Weeks SUM 1                  58407
Gt 101 To 102 Weeks SUM 1                  58406
Gt 97 To 98 Weeks SUM 1                    58405
Gt 85 To 86 Weeks SUM 1                    58404
Gt 95 To 96 Weeks SUM 1                    58404
Gt 82 To 83 Weeks SUM 1                    58404
Gt 102 To 103 Weeks SUM 1                  58404
Gt 96 To 97 Weeks SUM 1                    58404
Gt 99 To 100 Weeks SUM 1                   58404
Gt 84 To 85 Weeks SUM 1                    58403
Gt 88 To 89 Weeks SUM 1                    58402
Gt 92 To 93 Weeks SUM 1                    58402
Gt 91 To 92 Weeks SUM 1                    58402
Gt 87 To 88 Weeks SUM 1                    58401
Gt 81 To 82 Weeks SUM 1                    58401
Gt 98 To 99 Weeks SUM 1                    58401
Gt 93 To 94 Weeks SUM 1                    58400
Gt 90 To 91 Weeks SUM 1                    58400
dtype: int64

## 7. Cross-period schema consistency

Read the header of every extract and compare against the reference schema. Stable schemas across periods are a precondition for stacking the extracts during ingestion.

In [9]:
reference_cols = list(df.columns)
mismatches = {}

for path in extracts:
    cols = list(pd.read_csv(path, nrows=0).columns)
    if cols != reference_cols:
        mismatches[path.name] = {
            "n_columns": len(cols),
            "missing_vs_reference": [c for c in reference_cols if c not in cols][:5],
            "extra_vs_reference": [c for c in cols if c not in reference_cols][:5],
        }

print(f"Reference schema: {len(reference_cols)} columns (from {sample_file.name})")
print(f"Extracts checked: {len(extracts)}")
print(f"Schema mismatches: {len(mismatches)}")
mismatches if mismatches else "All extracts share an identical schema."

Reference schema: 121 columns (from 20260331-RTT-March-2026-full-extract.csv)
Extracts checked: 24
Schema mismatches: 0


'All extracts share an identical schema.'

## 8. Combined cross-period view (lightweight)

Build a single in-memory DataFrame (`df_combined`) spanning all periods, restricted to identifier and summary-measure columns (`COMBINE_COLS`) to keep memory low. This enables cross-period profiling — e.g. monthly totals across the full two-year window — without loading the full ~120-column extracts.

This view is exploratory only; the persisted, cleansed unified dataset is produced in the ingestion step (notebook 02).

In [10]:
df_combined = pd.concat(
    (pd.read_csv(p, usecols=COMBINE_COLS, low_memory=False) for p in extracts),
    ignore_index=True,
)

# Derive a month-start timestamp from the 'RTT-<Month>-<Year>' period label.
df_combined["period_date"] = pd.to_datetime(
    df_combined["Period"].str.removeprefix("RTT-"), format="%B-%Y"
)

print(f"Combined rows:    {len(df_combined):,}")
print(f"Distinct periods: {df_combined['period_date'].nunique()}")
print(f"Memory:           {df_combined.memory_usage(deep=True).sum() / 1_000_000:.1f} MB")
df_combined.head()

Combined rows:    4,416,338
Distinct periods: 24
Memory:           815.1 MB


,Period,Provider Org Code,Provider Org Name,RTT Part Description,Treatment Function Code,Treatment Function Name,Total,Total All,period_date
0,RTT-April-2024,A4M8P,BUCKSHAW HOSPITAL,Completed Pathways For Non-Admitted Patients,C_101,Urology Service,1.0,1,2024-04-01
1,RTT-April-2024,A4M8P,BUCKSHAW HOSPITAL,Completed Pathways For Non-Admitted Patients,C_999,Total,1.0,1,2024-04-01
2,RTT-April-2024,A4M8P,BUCKSHAW HOSPITAL,Incomplete Pathways,C_100,General Surgery Service,NaN,1,2024-04-01
3,RTT-April-2024,A4M8P,BUCKSHAW HOSPITAL,Incomplete Pathways,C_101,Urology Service,NaN,2,2024-04-01
4,RTT-April-2024,A4M8P,BUCKSHAW HOSPITAL,Incomplete Pathways,C_110,Trauma and Orthopaedic Service,NaN,1,2024-04-01


In [11]:
# Summary across all periods (text and numeric columns).
df_combined.describe(include="all")

,Period,Provider Org Code,Provider Org Name,RTT Part Description,Treatment Function Code,Treatment Function Name,Total,Total All,period_date
count,4416338,4416338,4416338,4416338,4416338,4416338,1.229622e+06,4.416338e+06,4416338
unique,24,558,574,5,24,24,NaN,NaN,NaN
top,RTT-October-2024,RJ1,GUY'S AND ST THOMAS' NHS FOUNDATION TRUST,Incomplete Pathways,C_999,Total,NaN,NaN,NaN
freq,190138,94328,94328,1573853,962866,962866,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,5.958808e+01,1.291810e+02,2025-03-16 03:09:12.865835
min,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,1.000000e+00,2024-04-01 00:00:00
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,1.000000e+00,2024-09-01 00:00:00
50%,NaN,NaN,NaN,NaN,NaN,NaN,2.000000e+00,2.000000e+00,2025-03-01 00:00:00
75%,NaN,NaN,NaN,NaN,NaN,NaN,1.300000e+01,1.400000e+01,2025-09-01 00:00:00
max,NaN,NaN,NaN,NaN,NaN,NaN,2.005900e+04,1.447420e+05,2026-03-01 00:00:00


In [12]:
# Total waiting list size per month, summed across all providers, specialties and pathway types.
monthly_totals = df_combined.groupby("period_date")["Total All"].sum().sort_index()
monthly_totals

period_date
2024-04-01    24054106
2024-05-01    24250722
2024-06-01    23984500
2024-07-01    24611558
2024-08-01    23816822
2024-09-01    24003318
2024-10-01    24634334
2024-11-01    24003122
2024-12-01    23142112
2025-01-01    24151132
2025-02-01    23545342
2025-03-01    23940002
2025-04-01    23510512
2025-05-01    23652998
2025-06-01    23936148
2025-07-01    24093182
2025-08-01    22981852
2025-09-01    23831946
2025-10-01    24151636
2025-11-01    23066694
2025-12-01    22876804
2026-01-01    23410238
2026-02-01    23153438
2026-03-01    23704266
Name: Total All, dtype: int64

## 9. Findings

_To be completed after execution. Record: confirmed grain, dimensions to retain, pathway type(s) in scope (e.g. Incomplete Pathways), data-quality issues for the Silver layer, and the 18-week boundary used for breach derivation._